# Xem kết quả từ file `.joblib`

Notebook này dùng để:
- nạp file kết quả đã lưu bằng `joblib`
- hiển thị accuracy
- hiển thị trọng số ensemble
- hiển thị classification report
- vẽ confusion matrix

> Để notebook chạy đúng, hãy đặt file `.ipynb` này cùng thư mục với file `.joblib`.


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

joblib_path = "max_fusion_results.joblib"

if not os.path.exists(joblib_path):
    raise FileNotFoundError(f"Không tìm thấy file: {joblib_path}")

result = joblib.load(joblib_path)
print("Đã load thành công:", joblib_path)
print("Các khóa trong file:", list(result.keys()))


In [ ]:
acc = result.get("acc_ens", None)
weights = result.get("weights", {})
classes = result.get("classes", [])
cm = result.get("cm", None)
report = result.get("report", {})

print(f"Accuracy ensemble: {acc:.4f}" if acc is not None else "Không có acc_ens")
print("Số lớp:", len(classes))


In [ ]:
weights_df = pd.DataFrame(
    list(weights.items()), columns=["Feature / Model", "Weight"]
).sort_values("Weight", ascending=False)

weights_df


In [ ]:
if isinstance(report, dict) and len(report) > 0:
    report_df = pd.DataFrame(report).T
    report_df
else:
    print("Không có classification report dạng dict để hiển thị.")


In [ ]:
if cm is not None:
    cm = np.array(cm)
    print("Kích thước confusion matrix:", cm.shape)
    print(cm)
else:
    print("Không có confusion matrix.")


In [ ]:
if cm is not None:
    plt.figure(figsize=(10, 8))
    plt.imshow(cm, interpolation="nearest")
    plt.title("Confusion Matrix")
    plt.colorbar()

    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=90)
    plt.yticks(tick_marks, classes)

    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(
                j, i, str(cm[i, j]),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=8
            )

    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.show()


In [ ]:
summary = {
    "accuracy": float(acc) if acc is not None else None,
    "num_classes": int(len(classes)),
    "classes": list(classes),
    "weights": weights,
}
summary


In [ ]:
import joblib
from sklearn.metrics import classification_report
import re

# load file
res = joblib.load("max_fusion_results.joblib")

print("== LOAD RESULTS ==")

# -------- Ensemble Accuracy --------
acc = res["acc_ens"]
print(f"\nEnsemble Accuracy: {acc:.4f}")

# -------- Parse classification report --------
report_str = res["report"]
print("\nRaw Report:\n")
print(report_str)

# extract macro & weighted bằng regex (vì đang là string)
macro_line = re.search(r"macro avg\s+([\d\.]+)\s+([\d\.]+)\s+([\d\.]+)", report_str)
weighted_line = re.search(r"weighted avg\s+([\d\.]+)\s+([\d\.]+)\s+([\d\.]+)", report_str)

macro_precision = float(macro_line.group(1))
macro_recall = float(macro_line.group(2))
macro_f1 = float(macro_line.group(3))

weighted_f1 = float(weighted_line.group(3))

# -------- Print Table 3 --------
print("\n== TABLE 3: OVERALL PERFORMANCE ==")
print(f"{'Metric':<20} {'Value'}")
print("-"*30)
print(f"{'Ensemble accuracy':<20} {acc:.4f}")
print(f"{'Macro precision':<20} {macro_precision:.4f}")
print(f"{'Macro recall':<20} {macro_recall:.4f}")
print(f"{'Macro F1-score':<20} {macro_f1:.4f}")
print(f"{'Weighted F1-score':<20} {weighted_f1:.4f}")